In [1]:
import csv, sys
import pandas as pd

csv.field_size_limit(sys.maxsize)  # alguns textos de checagem passam do limite padrão do módulo csv

with open("data/factchecksbr/central_de_fatos.tsv", encoding="utf-8") as f:
    leitor = csv.reader(f, delimiter="\t")
    colunas = next(leitor)
    cdf = pd.DataFrame([dict(zip(colunas, linha)) for linha in leitor])


print(cdf.shape)
print(cdf.columns.tolist())


(10461, 8)
['review_id', 'review_text', 'review_author', 'review_url', 'review_domain', 'review_date', 'category', 'is_fake']


In [2]:
# Na Central de Fatos, o título da checagem é a 1ª linha do review_text
titulo = cdf["review_text"].str.split("\n").str[0].str.strip()
titulo.head(10)


0    Publicações enganam ao associar Bolsonaro à ap...
1    Post mostra imagens de outras estradas para af...
2    Eduardo Bolsonaro posta vídeo antigo sobre lib...
3    Médica cita estudos não conclusivos para suger...
4    Médica usa informações falsas em vídeo para fa...
5    Imagem de Doria tomando vacina é de março, ant...
6    Site engana ao afirmar que baixa mortalidade p...
7    Vacinas para covid-19 que chegaram ao Brasil s...
8    Tuíte engana ao afirmar que vacinas usam célul...
9    Brasil não terá uma nova moeda lastreada em ni...
Name: review_text, dtype: object

In [3]:
for padrao in [r"veja o que", r"\d+\s+boatos", r"o\s+melhor"]:
    pegos = titulo[titulo.str.match(padrao, case=False)]
    print(f"\n=== {padrao}: {len(pegos)} títulos")
    print(pegos.head(5).to_string())



=== veja o que: 2 títulos
977    Veja o que é #FATO ou #FAKE no discurso de Bol...
990    Veja o que é #FATO ou #FAKE em vídeo do horári...

=== \d+\s+boatos: 41 títulos
1790    7 boatos sobre desenhos animados e animes que ...
1808    7 boatos que surgiram na web após o anúncio da...
1840    7 boatos envolvendo o carnaval que já enganara...
2078    7 boatos sobre o “chip da besta” que vocês pre...
2096    7 boatos que surgiram em áudios no WhatsApp e ...

=== o\s+melhor: 25 títulos
2213    O melhor de 2020 no Boatos.org, por Edgard Mat...
2215       O melhor do Boatos.org em 2020, por Carol Lira
2219    O melhor do Boatos.org em 2020, por Raiane Gonoli
2221     O melhor do Boatos.org em 2020, por Kyene Becker
2908    O melhor do Boatos.org em 2017, por Edgard Mat...


In [4]:
from limpeza import limpar_titulo, eh_multi_alegacao

cdf["titulo"] = titulo
cdf["claim"] = limpar_titulo(cdf["titulo"])
cdf["multi_claim"] = eh_multi_alegacao(cdf["titulo"])

print("títulos alterados:", (cdf["titulo"] != cdf["claim"]).sum())
print("multi-alegação:", cdf["multi_claim"].sum())
print("claim vazio:", (cdf["claim"] == "").sum())
cdf.loc[cdf["titulo"] == cdf["claim"], "titulo"].sample(15, random_state=42)


títulos alterados: 8106
multi-alegação: 69
claim vazio: 4


8238    Capa do New York Times que exalta manifestaçõe...
860     Não, China não coleta DNA dos voluntários de t...
889     Boato usa fotos antigas para alertar sobre che...
8453    Livro infantil que trata de incesto não foi di...
6997    Grupo terrorista Vale/Samarco assume atentado ...
8341    Foto de manifestante que pede fim de universid...
501     Posts que anunciam vitória de Bolsonaro no 1º ...
139     Postagens sobre transposição do São Francisco ...
7000    Dilma declara guerra ao Islã e ao Estado Islâm...
8336    Fox News não veiculou pesquisa que indica vitó...
8499    Lula não disse que compraria votos de baianos ...
5239        Filho de Edir Macedo é gay, diz notícia falsa
945     Vaias em evento do PT não foram para Haddad, m...
7379    Retrospectiva 2015: Todas as declarações menti...
2473    Lorota: massagista americana gasta R$ 47 mil p...
Name: titulo, dtype: object

In [5]:
import numpy as np

# Antigo: alegações válidas, com ano
cdf["ano"] = pd.to_datetime(cdf["review_date"]).dt.year
antigo = cdf[~cdf["multi_claim"] & (cdf["claim"] != "")].reset_index(drop=True)

# 2022: o claim não mudou com a limpeza nova (o diff deu 0), então os embeddings do 01 continuam válidos
d22 = pd.read_csv("data/com_texto_limpo.csv")
E22_todos = np.load("data/emb_minilm.npy")
assert len(d22) == len(E22_todos)
manter = (~d22["multi_claim"] & pd.to_datetime(d22["Data da checagem"]).notna()).to_numpy()
d22, E22 = d22[manter].reset_index(drop=True), E22_todos[manter]
print(len(antigo), len(d22))


10388 1844


In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                            revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42")
E_antigo = model.encode(antigo["claim"].tolist(), normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)
np.save("data/factchecksbr/emb_central_minilm.npy", E_antigo)


/Users/aluno2/Residencia/RES-IA-Challenge-1/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 163/163 [00:16<00:00,  9.72it/s]


In [7]:
TAUS = [0.80, 0.85, 0.90, 0.95]          # grade pré-registrada no design.md (commit 3294219)

S = E22 @ E_antigo.T                      # 1.844 × ~10.388: cada linha é uma alegação de 2022 contra o histórico
melhor = S.argmax(axis=1)
d22["sim_max"] = S.max(axis=1)
d22["vizinho"] = antigo["claim"].to_numpy()[melhor]
d22["ano_vizinho"] = antigo["ano"].to_numpy()[melhor]

e2018 = (antigo["ano"] == 2018).to_numpy()
d22["sim_max_2018"] = S[:, e2018].max(axis=1)

grade = pd.DataFrame({
    "τ": TAUS,
    "taxa_todos_anos": [(d22["sim_max"] >= t).mean() for t in TAUS],
    "taxa_so_2018": [(d22["sim_max_2018"] >= t).mean() for t in TAUS],
})
grade

,τ,taxa_todos_anos,taxa_so_2018
0,0.80,0.116594,0.058568
1,0.85,0.037961,0.015184
2,0.90,0.015727,0.005965
3,0.95,0.002169,0.001085


In [8]:
from pathlib import Path
RESULTS = Path("results")
FAIXAS = [(0.80, 0.85), (0.85, 0.90), (0.90, 0.95), (0.95, 1.01)]
POR_FAIXA = 20
rng = np.random.default_rng(44)   # semente do pré-registro

d22["idx_antigo"] = melhor
pares, usadas_antigo = [], set()
for lo, hi in FAIXAS:
    na_faixa = d22.index[(d22["sim_max"] >= lo) & (d22["sim_max"] < hi)].to_numpy()
    escolhidos = 0
    for i in rng.permutation(na_faixa):
        if escolhidos >= POR_FAIXA:
            break
        j = d22.at[i, "idx_antigo"]
        if j in usadas_antigo:      # cada alegação antiga em no máximo um par
            continue
        usadas_antigo.add(j)
        escolhidos += 1
        pares.append({"faixa": f"{lo:.2f}-{min(hi, 1):.2f}", "sim": round(float(d22.at[i, "sim_max"]), 3),
                      "idx_2022": d22.at[i, "idx"] if "idx" in d22 else i, "idx_antigo": int(j),
                      "alegacao_2022": d22.at[i, "claim"], "alegacao_antiga": antigo.at[j, "claim"],
                      "ano_antiga": int(antigo.at[j, "ano"])})
    print(f"{lo:.2f}-{min(hi, 1):.2f}: {escolhidos} pares (de {len(na_faixa)} alegações na faixa)")

pares = pd.DataFrame(pares).sample(frac=1, random_state=44).reset_index(drop=True)  # embaralha as faixas
pares.insert(0, "par_id", range(1, len(pares) + 1))

caminho_rotulos = RESULTS / "validacao_entre_ciclos.csv"
if not caminho_rotulos.exists():   # não sobrescreve rótulos já feitos
    (pares[["par_id", "alegacao_2022", "alegacao_antiga", "ano_antiga"]]
     .assign(rotulo="", tipo="")
     .to_csv(caminho_rotulos, index=False))
pares[["par_id", "faixa", "sim", "idx_2022", "idx_antigo"]].to_csv(RESULTS / "validacao_entre_ciclos_chave.csv", index=False)
print(f"\nRotular: {caminho_rotulos} ({len(pares)} pares). NÃO abra o arquivo _chave antes de terminar.")


0.80-0.85: 20 pares (de 145 alegações na faixa)
0.85-0.90: 20 pares (de 41 alegações na faixa)
0.90-0.95: 18 pares (de 25 alegações na faixa)
0.95-1.00: 1 pares (de 4 alegações na faixa)

Rotular: results/validacao_entre_ciclos.csv (59 pares). NÃO abra o arquivo _chave antes de terminar.
